# exp_2 on Kaggle: two full runs on 2x T4

Runs `B_strat_s0` (GPU 0) and `L_flip_on_D_la_s0` (GPU 1) with the exp_2 code from GitHub, 16k iterations (= the 16k checkpoint of the author's schedule), then exports all checkpoints with the mirror / deskew probes and packs everything into `/kaggle/working/exp2_kaggle_results.tgz`.

**Before "Save & Run All (Commit)":**
- Settings: Accelerator **GPU T4 x2**, Internet **on**, Persistence off.
- Add Input: your private dataset **Scoliosis_1K**.
- Only if the dataset holds the password-protected zip: Add-ons -> Secrets -> add `SCOLIOSIS_ZIP_PASSWORD`.
- Keep this notebook **private**: the output contains logits of the licensed dataset.

Kaggle stops a session after 12 hours, so training is stopped at `TRAIN_HOURS` and the export uses whatever checkpoints exist.

In [ ]:
RUNS = {'B_strat_s0': 0, 'L_flip_on_D_la_s0': 1}   # run -> GPU
TOTAL_ITER = 16000
TRAIN_HOURS = 10.3                                  # leave time for export + packing inside Kaggle's 12 h
REPO_URL = 'https://github.com/dindawijayanto/scoliosis-gait-reproduction.git'

import os, glob, subprocess, time, shutil
START = time.time()
def sh(cmd, check=True):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-4000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r.stdout

sh('nvidia-smi --query-gpu=index,name,memory.total --format=csv')

step 1: code and packages

In [ ]:
os.chdir('/kaggle/working')
if not os.path.exists('scoliosis-gait-reproduction'):
    sh(f'git clone -q {REPO_URL}')
EXP = '/kaggle/working/scoliosis-gait-reproduction/checkpoint_2/exp_2'
os.chdir(EXP)
sh('git log --oneline -1')
sh('pip install -q kornia einops imageio fvcore lightly tensorboard opencv-python-headless 2>&1 | tail -2', check=False)
sh('python -c "import sys; sys.path.insert(0, \'opengait\'); from modeling import models; print(\'import OK\', hasattr(models, \'ScoNetRobust\'))"')

step 2: find the data (extracted folder, or the password zip -> /tmp, never /kaggle/working)

In [ ]:
def find_pkl_root():
    for root, dirs, files in os.walk('/kaggle/input'):
        if len([d for d in dirs if d.isdigit() and len(d) == 5]) > 1400:
            return root
    return None

DATA = find_pkl_root()
if DATA is None:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'no extracted Scoliosis1K folder and no zip under /kaggle/input'
    from kaggle_secrets import UserSecretsClient
    pw = UserSecretsClient().get_secret('SCOLIOSIS_ZIP_PASSWORD')
    os.makedirs('/tmp/data', exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', '-P', pw, zips[0], '-d', '/tmp/data'], check=True)
    DATA = glob.glob('/tmp/data/**/Scoliosis1K-sil-pkl', recursive=True)[0]
n = len(glob.glob(f'{DATA}/*/*/*/*.pkl'))
print('data:', DATA, '| pkl sequences:', n, '(expected 1493)')
assert n == 1493

step 3: configs (16k, 2 data workers per run: Kaggle has 4 CPUs)

In [ ]:
sh(f'python exp_2/make_configs.py --data "{DATA}"')
sh(f"sed -i 's/total_iter: 20000/total_iter: {TOTAL_ITER}/; s/num_workers: 8/num_workers: 2/' exp_2/configs/*.yaml")
sh('grep -m1 total_iter exp_2/configs/B_strat_s0.yaml; grep -m1 train_flip exp_2/configs/L_flip_on_D_la_s0.yaml')

step 4: train both runs at the same time, one per GPU, with a time limit

In [ ]:
os.makedirs('exp_2/logs', exist_ok=True)
procs = {}
for i, (run, gpu) in enumerate(RUNS.items()):
    env = dict(os.environ, OPENGAIT_SEED=run.rsplit('_s', 1)[1], CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = ['torchrun', '--nproc_per_node=1', f'--master_port={29610 + i}', 'opengait/main.py',
           '--cfgs', f'exp_2/configs/{run}.yaml', '--phase', 'train', '--log_to_file']
    procs[run] = subprocess.Popen(cmd, env=env, stdout=open(f'exp_2/logs/{run}.log', 'w'), stderr=subprocess.STDOUT)
    print('started', run, 'on GPU', gpu)

def last_iter_line(run):
    lines = [l for l in open(f'exp_2/logs/{run}.log', errors='ignore') if 'Iteration' in l]
    return lines[-1].strip()[:140] if lines else '(no iteration yet)'

while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    hours = (time.time() - START) / 3600
    for run in procs:
        print(f'{hours:5.2f} h  {run}: {last_iter_line(run)}', flush=True)
    if hours > TRAIN_HOURS:
        print('time limit reached: stopping training, exporting the checkpoints that exist')
        for p in procs.values():
            p.terminate()
        break
for run, p in procs.items():
    p.wait()
    print(run, 'exit code', p.returncode)
    sh(f'grep -m3 -E "Error|error:" exp_2/logs/{run}.log', check=False)
sh('ls output/Scoliosis1K/ScoNetRobust/*/checkpoints/')

step 5: export (all checkpoints + mirror / deskew probes + embeddings), both runs in parallel

In [ ]:
ex = {run: subprocess.Popen(f'python exp_2/export_logits.py --runs {run} > exp_2/logs/export_{run}.log 2>&1',
                            shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
      for run, gpu in RUNS.items()}
for run, p in ex.items():
    p.wait()
    sh(f'grep -E "iter|ALL DONE|Error" exp_2/logs/export_{run}.log | tail -12', check=False)
sh('python exp_2/calibrate.py --ref none', check=False)

step 6: pack the results (download this file from the Output tab)

In [ ]:
cks = ' '.join(glob.glob('output/Scoliosis1K/ScoNetRobust/*/checkpoints/*-16000.pt'))
sh(f'tar czf /kaggle/working/exp2_kaggle_results.tgz exp_2/results exp_2/logits exp_2/logs exp_2/configs {cks}')
sh('ls -lh /kaggle/working/exp2_kaggle_results.tgz')
shutil.rmtree('/kaggle/working/scoliosis-gait-reproduction/output', ignore_errors=True)   # keep only the tgz as output
print(f'done in {(time.time() - START) / 3600:.2f} h')